In [5]:
import json
import pandas as pd
import statistics
from collections import defaultdict
from pathlib import Path
from datetime import datetime, timezone

# --- Anchor systems for each dataset ---
ANCHOR_SYSTEMS_A = ['財務会計システム', 'HR人事給与システム', '受発注在庫管理システム', 'ProcMine SSO']

ANCHOR_SYSTEMS_B = [
    '財務会計システム', 'HR人事給与システム', '受発注在庫管理システム',
    'shinkuitorihikisaki_touroku_tetsuzuki',
    'gyomu_itaku_keihi_kitei',
    'shinkui_keiyaku_tetsuzuki',
    'getsujitsu_teigaku_torihikisaki_ichiran',
    'nyusha_checklist_shinsotsu_batch',
    'keiyaku_kaijo_tetsuzuki',
    'gyomu_itaku_ukeire_tetsuzuki',
    'kanrisya_kengen_shinsei_tetsuzuki',
    'ikuji_kyuugyou_kitei',
]

LABEL_TRANSLATIONS = {
    '財務会計システム': 'financial_accounting',
    'HR人事給与システム': 'hr_payroll',
    '受発注在庫管理システム': 'order_inventory',
    'ProcMine SSO': 'procmine_sso',
    'shinkuitorihikisaki_touroku_tetsuzuki': 'new_supplier_registration',
    'gyomu_itaku_keihi_kitei': 'outsourcing_expense_regulations',
    'shinkui_keiyaku_tetsuzuki': 'new_supplier_contract',
    'getsujitsu_teigaku_torihikisaki_ichiran': 'monthly_fixed_transaction_list',
    'nyusha_checklist_shinsotsu_batch': 'new_hire_checklist',
    'keiyaku_kaijo_tetsuzuki': 'contract_termination',
    'gyomu_itaku_ukeire_tetsuzuki': 'outsourcing_acceptance',
    'kanrisya_kengen_shinsei_tetsuzuki': 'admin_authority_application',
    'ikuji_kyuugyou_kitei': 'childcare_leave_regulations',
}

CLIPBOARD_MIN_GAP_S = 5
LONG_SEGMENT_THRESHOLD_MS = 60000
EXCEPTION_MULTIPLIER = 1.8
MIN_DURATION_FOR_VARIANT_S = 3


def load_session_events(session_dir):
    session_dir = Path(session_dir)
    all_events = []
    chunk_dirs = [d for d in session_dir.iterdir() if d.is_dir() and d.name.startswith("chunk_")]
    for chunk_dir in chunk_dirs:
        events_path = chunk_dir / "events.jsonl"
        if events_path.exists():
            with open(events_path, "r", encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if line:
                        all_events.append(json.loads(line))
    all_events.sort(key=lambda e: e["timestamp_ms"])
    return all_events


def get_anchor(title, anchor_systems):
    for a in anchor_systems:
        if a in title:
            return a
    return None


def filter_noise(app_switches_df):
    kept, last = [], None
    for _, row in app_switches_df.iterrows():
        ident = (row['process_id'], row['window_title'])
        if ident != last:
            kept.append(row)
            last = ident
    return pd.DataFrame(kept)


def build_anchor_segments(clean_df, session_end_ms, anchor_systems):
    if len(clean_df) == 0:
        return []
    segments = []
    seg_start_ms = int(clean_df.iloc[0]['timestamp_iso'].timestamp() * 1000)
    current_anchor = None
    for i in range(len(clean_df)):
        title = clean_df.iloc[i]['window_title']
        anchor = get_anchor(title, anchor_systems)
        if anchor is not None:
            if current_anchor is None:
                current_anchor = anchor
            elif anchor != current_anchor:
                seg_end_ms = int(clean_df.iloc[i]['timestamp_iso'].timestamp() * 1000)
                segments.append({'start_ms': seg_start_ms, 'end_ms': seg_end_ms, 'label': current_anchor})
                seg_start_ms = seg_end_ms
                current_anchor = anchor
    segments.append({'start_ms': seg_start_ms, 'end_ms': session_end_ms, 'label': current_anchor})
    return segments


def refine_with_clipboard(segments, clipboard_ts_list):
    refined = []
    for seg in segments:
        duration = seg['end_ms'] - seg['start_ms']
        label = seg.get('label', 'unknown')
        if duration <= LONG_SEGMENT_THRESHOLD_MS:
            refined.append(seg)
            continue
        inside = [t for t in clipboard_ts_list if seg['start_ms'] < t < seg['end_ms']]
        split_points = []
        last_kept = None
        for t in inside:
            if last_kept is None or (t - last_kept) > CLIPBOARD_MIN_GAP_S * 1000:
                split_points.append(t)
                last_kept = t
        if not split_points:
            refined.append(seg)
            continue
        prev = seg['start_ms']
        for sp in split_points:
            refined.append({'start_ms': prev, 'end_ms': sp, 'label': label})
            prev = sp
        refined.append({'start_ms': prev, 'end_ms': seg['end_ms'], 'label': label})
    return refined


def translate_labels(segments):
    for s in segments:
        s['label'] = LABEL_TRANSLATIONS.get(s['label'], s['label'])
    return segments


def tag_variants(segments, full_df):
    ts_ms_col = full_df['timestamp_iso'].apply(lambda t: int(t.timestamp() * 1000))

    def count_actions(start_ms, end_ms):
        mask = (ts_ms_col >= start_ms) & (ts_ms_col <= end_ms)
        subset = full_df[mask]
        counts = subset['event_type'].value_counts()
        return (counts.get('keystroke', 0) + counts.get('mouse_click', 0) +
                counts.get('shortcut', 0) + counts.get('mouse_scroll', 0))

    for seg in segments:
        duration_s = (seg['end_ms'] - seg['start_ms']) / 1000
        seg['action_density'] = count_actions(seg['start_ms'], seg['end_ms']) / duration_s if duration_s > 0 else 0
        seg['duration_s'] = duration_s

    density_by_label = defaultdict(list)
    for seg in segments:
        if seg['duration_s'] >= MIN_DURATION_FOR_VARIANT_S:
            density_by_label[seg['label']].append(seg['action_density'])

    median_density = {label: statistics.median(d) for label, d in density_by_label.items() if d}

    for seg in segments:
        if seg['duration_s'] < MIN_DURATION_FOR_VARIANT_S:
            continue
        med = median_density.get(seg['label'], 0)
        if med > 0 and seg['action_density'] > med * EXCEPTION_MULTIPLIER:
            seg['label'] = seg['label'] + '_exception'
    return segments


def process_session(session_dir, session_id, anchor_systems):
    raw_events = load_session_events(session_dir)
    if not raw_events:
        return []

    df = pd.DataFrame(raw_events)
    df['timestamp_iso'] = pd.to_datetime(df['timestamp_iso'])

    app_switches = df[df['event_type'] == 'app_switch'].copy()
    app_switches['process_id'] = app_switches['payload'].apply(lambda p: p['new_app']['process_id'])
    app_switches['window_title'] = app_switches['payload'].apply(lambda p: p['new_app']['window_title'])

    clean = filter_noise(app_switches).reset_index(drop=True)
    if len(clean) == 0:
        return []

    session_end_ms = int(df['timestamp_iso'].max().timestamp() * 1000)
    segments = build_anchor_segments(clean, session_end_ms, anchor_systems)

    clipboard_events = df[df['event_type'] == 'clipboard_change'].copy()
    clipboard_ts_list = [int(t.timestamp() * 1000) for t in clipboard_events['timestamp_iso']]
    segments = refine_with_clipboard(segments, clipboard_ts_list)

    segments = translate_labels(segments)
    segments = tag_variants(segments, df)

    output = []
    for s in segments:
        output.append({
            'session_id': session_id,
            'start': datetime.fromtimestamp(s['start_ms'] / 1000, tz=timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ'),
            'end': datetime.fromtimestamp(s['end_ms'] / 1000, tz=timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ'),
            'label': s['label'],
        })
    return output
# RUN ACROSS ALL SESSIONS IN dataset_b (NO ground truth to check against)

dataset_b_path = Path("../data/dataset_b")
all_segments_output_b = []

session_dirs = sorted([d for d in dataset_b_path.iterdir() if d.is_dir()])
print(f"Found {len(session_dirs)} sessions in Dataset B")

for session_dir in session_dirs:
    session_id = session_dir.name
    try:
        segments = process_session(session_dir, session_id, ANCHOR_SYSTEMS_B)
        all_segments_output_b.extend(segments)
        print(f"{session_id}: {len(segments)} segments")
    except Exception as e:
        print(f"ERROR on {session_id}: {e}")

print(f"\nTotal segments generated across Dataset B: {len(all_segments_output_b)}")

Found 15 sessions in Dataset B
ses_20260701-164424-CHAITANYA0BCF: 60 segments
ses_20260701-171614-CHAITANYA0BCF: 72 segments
ses_20260701-173246-SIDDHIGUPTAB00B: 64 segments
ses_20260701-173642-NEELA9BAF: 39 segments
ses_20260701-175258-LAPTOP-76QMG9DE: 55 segments
ses_20260701-175747-SIDDHIGUPTAB00B: 52 segments
ses_20260701-180923-NEELA9BAF: 70 segments
ses_20260701-181413-LAPTOP-76QMG9DE: 72 segments
ses_20260701-181913-SIDDHIGUPTAB00B: 70 segments
ses_20260701-182634-NEELA9BAF: 53 segments
ses_20260701-183232-LAPTOP-76QMG9DE: 31 segments
ses_20260701-184201-LAPTOP-76QMG9DE: 90 segments
ses_20260701-190250-NEELA9BAF: 47 segments
ses_20260701-191537-LAPTOP-76QMG9DE: 48 segments
ses_20260701-192455-NEELA9BAF: 45 segments

Total segments generated across Dataset B: 868


In [6]:
import json

with open('../segments.jsonl', 'w', encoding='utf-8') as f:
    for seg in all_segments_output_b:
        f.write(json.dumps(seg, ensure_ascii=False) + '\n')

print(f"Wrote {len(all_segments_output_b)} segments to segments.jsonl")

Wrote 868 segments to segments.jsonl


In [7]:
import pandas as pd
from datetime import datetime

seg_df = pd.DataFrame(all_segments_output_b)
seg_df['start_dt'] = pd.to_datetime(seg_df['start'])
seg_df['end_dt'] = pd.to_datetime(seg_df['end'])
seg_df['duration_s'] = (seg_df['end_dt'] - seg_df['start_dt']).dt.total_seconds()

# Group by label: frequency, avg/median duration, number of distinct sessions (proxy for workers)
summary = seg_df.groupby('label').agg(
    frequency=('label', 'count'),
    avg_duration_s=('duration_s', 'mean'),
    median_duration_s=('duration_s', 'median'),
    total_time_s=('duration_s', 'sum'),
    distinct_sessions=('session_id', 'nunique'),
).sort_values('total_time_s', ascending=False)

print(summary)

                                           frequency  avg_duration_s  \
label                                                                  
hr_payroll                                       247       13.331984   
financial_accounting                             168       16.422619   
order_inventory                                  136       14.419118   
financial_accounting_exception                    20       22.950000   
new_supplier_contract                             44        6.750000   
monthly_fixed_transaction_list                    55        5.036364   
new_hire_checklist                                39        6.794872   
new_supplier_registration                         44        5.522727   
outsourcing_expense_regulations                   31        6.129032   
outsourcing_acceptance                            18        9.333333   
new_supplier_registration_exception                8       13.375000   
contract_termination                              14        7.14

In [8]:
# Labels that came from shared "room" browser systems (exclude these from automation ranking)
ROOM_LABELS = {'hr_payroll', 'financial_accounting', 'order_inventory',
               'hr_payroll_exception', 'financial_accounting_exception', 'order_inventory_exception'}

# Reset index so 'label' becomes a normal column again
summary_reset = summary.reset_index()

# Split into "specific document" candidates vs "room" labels
document_labels = summary_reset[~summary_reset['label'].isin(ROOM_LABELS)].copy()
room_labels = summary_reset[summary_reset['label'].isin(ROOM_LABELS)].copy()

# Separate base process name from its exception variant, so we can compute exception RATE per process
document_labels['base_process'] = document_labels['label'].str.replace('_exception', '', regex=False)
document_labels['is_exception'] = document_labels['label'].str.endswith('_exception')

# Build a combined view: for each base process, total frequency/time across standard + exception,
# plus what fraction of its executions were exceptions (risk indicator)
combined = document_labels.groupby('base_process').agg(
    total_frequency=('frequency', 'sum'),
    total_time_s=('total_time_s', 'sum'),
    total_distinct_sessions=('distinct_sessions', 'max'),  # approx - max avoids double-counting overlap
).reset_index()

exception_counts = document_labels[document_labels['is_exception']].set_index('base_process')['frequency']
combined['exception_count'] = combined['base_process'].map(exception_counts).fillna(0)
combined['exception_rate'] = combined['exception_count'] / combined['total_frequency']

# Priority score: total time cost, discounted by exception rate (higher exception rate = riskier to automate)
combined['priority_score'] = combined['total_time_s'] * (1 - combined['exception_rate'])

final_ranking = combined.sort_values('priority_score', ascending=False)

print("=== AUTOMATION PRIORITY RANKING (specific-document processes only) ===\n")
print(final_ranking[['base_process', 'total_frequency', 'total_time_s',
                      'exception_rate', 'total_distinct_sessions', 'priority_score']].to_string(index=False))

print(f"\n(Excluded {len(room_labels)} 'room' label rows - these represent multiple mixed, unidentifiable "
      f"processes sharing a browser system, not single automatable tasks)")

=== AUTOMATION PRIORITY RANKING (specific-document processes only) ===

                   base_process  total_frequency  total_time_s  exception_rate  total_distinct_sessions  priority_score
          new_supplier_contract               50         385.0        0.120000                        6      338.800000
      new_supplier_registration               52         350.0        0.153846                       10      296.153846
 monthly_fixed_transaction_list               55         277.0        0.000000                        9      277.000000
             new_hire_checklist               42         286.0        0.071429                        9      265.571429
outsourcing_expense_regulations               36         249.0        0.138889                        7      214.416667
         outsourcing_acceptance               18         168.0        0.000000                        6      168.000000
           contract_termination               14         100.0        0.000000          

In [9]:
import pandas as pd

seg_df = pd.DataFrame(all_segments_output_b)
target_label = 'monthly_fixed_transaction_list'
examples = seg_df[seg_df['label'] == target_label].head(3)
print(examples)

                          session_id                 start  \
1  ses_20260701-164424-CHAITANYA0BCF  2026-07-01T16:45:00Z   
3  ses_20260701-164424-CHAITANYA0BCF  2026-07-01T16:45:16Z   
6  ses_20260701-164424-CHAITANYA0BCF  2026-07-01T16:45:51Z   

                    end                           label  
1  2026-07-01T16:45:04Z  monthly_fixed_transaction_list  
3  2026-07-01T16:45:19Z  monthly_fixed_transaction_list  
6  2026-07-01T16:45:55Z  monthly_fixed_transaction_list  


In [11]:
import json
from pathlib import Path
from datetime import datetime

def load_session_events_raw(session_dir):
    all_events = []
    chunk_dirs = [d for d in session_dir.iterdir() if d.is_dir() and d.name.startswith("chunk_")]
    for chunk_dir in chunk_dirs:
        events_path = chunk_dir / "events.jsonl"
        if events_path.exists():
            with open(events_path, "r", encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if line:
                        all_events.append(json.loads(line))
    all_events.sort(key=lambda e: e["timestamp_ms"])
    return all_events

dataset_b_path = Path("../data/dataset_b")
session_dirs = sorted([d for d in dataset_b_path.iterdir() if d.is_dir()])

# Search ALL sessions for extracted_text mentioning invoice confirmation comments
for session_dir in session_dirs:
    raw_events = load_session_events_raw(session_dir)
    for e in raw_events:
        extracted = e.get('context', {}).get('extracted_text')
        if extracted and isinstance(extracted, dict):
            text = extracted.get('text', '')
            if '請求書照合完了' in text:  # "invoice reconciliation complete"
                print(f"{session_dir.name} | {e['timestamp_iso']}")
                print(f"  {text}")
                print()

ses_20260701-164424-CHAITANYA0BCF | 2026-07-01T16:45:05.172Z
  請求書照合完了。INV-2026-7345　金額：455,128円。差異あり要確認。

ses_20260701-164424-CHAITANYA0BCF | 2026-07-01T16:47:12.144Z
  請求書照合完了。INV-2026-7350　金額：1,993,825円。差異あり要確認。

ses_20260701-164424-CHAITANYA0BCF | 2026-07-01T16:52:20.621Z
  請求書照合完了。INV-2026-7352　金額：393,384円。差異なし承認。

ses_20260701-164424-CHAITANYA0BCF | 2026-07-01T16:52:36.303Z
  請求書照合完了。INV-2026-7353　金額：437,452円。差異なし承認。

ses_20260701-164424-CHAITANYA0BCF | 2026-07-01T16:52:59.059Z
  請求書照合完了。INV-2026-7355　金額：1,685,384円。差異なし承認。

ses_20260701-175258-LAPTOP-76QMG9DE | 2026-07-01T18:02:02.453Z
  請求書照合完了。INV-2026-7008　金額：408,812円。差異あり要確認。

ses_20260701-175258-LAPTOP-76QMG9DE | 2026-07-01T18:02:54.568Z
  請求書照合完了。INV-2026-7011　金額：3,088,562円。差異なし承認。

ses_20260701-175747-SIDDHIGUPTAB00B | 2026-07-01T18:02:40.378Z
  請求書照合完了。INV-2026-7299　金額：530,056円。差異なし承認。

ses_20260701-175747-SIDDHIGUPTAB00B | 2026-07-01T18:02:59.560Z
  請求書照合完了。INV-2026-7300　金額：1,177,577円。差異あり要確認。

ses_20260701-175747-SIDDHI

In [12]:
import json
from datetime import datetime
from pathlib import Path

def load_session_events_raw(session_dir):
    all_events = []
    chunk_dirs = [d for d in session_dir.iterdir() if d.is_dir() and d.name.startswith("chunk_")]
    for chunk_dir in chunk_dirs:
        events_path = chunk_dir / "events.jsonl"
        if events_path.exists():
            with open(events_path, "r", encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if line:
                        all_events.append(json.loads(line))
    all_events.sort(key=lambda e: e["timestamp_ms"])
    return all_events

session_dir = Path("../data/dataset_b/ses_20260701-182634-NEELA9BAF")
raw_events = load_session_events_raw(session_dir)

# Look at a wider window around when "this list" instruction appeared
ws = datetime.fromisoformat("2026-07-01T18:29:30+00:00")
we = datetime.fromisoformat("2026-07-01T18:31:00+00:00")

for e in raw_events:
    ts = datetime.fromisoformat(e['timestamp_iso'].replace('Z', '+00:00'))
    if ws <= ts <= we:
        etype = e['event_type']
        if etype == 'app_switch':
            print(f"{e['timestamp_iso']}  APP_SWITCH -> {e['payload']['new_app'].get('window_title')}")
        extracted = e.get('context', {}).get('extracted_text')
        if extracted and isinstance(extracted, dict):
            text = extracted.get('text', '')
            print(f"{e['timestamp_iso']}  EXTRACTED_TEXT:\n{text}\n---")

2026-07-01T18:29:31.212Z  APP_SWITCH -> 財務会計システム - Profile 1 - Microsoft​ Edge
2026-07-01T18:29:31.212Z  EXTRACTED_TEXT:
受発
受発注在庫管理システム
2026年度 · 物流管理部
2026/07/01 23:59:30
高橋 健司 · 物流マネージャー
高
受発注在庫業務
ダッシュボード
在庫管理
12
契約管理
11
返品処理
0
IT申請
12
受注・その他
0
ダッシュボード / 契約管理
契約管理
12 件 · 10 件申請中
申請一覧
ID
社員ID
氏名
申請種別
期間・詳細
部署
ステータス
P12-07046495-001
V3002
大和商事株式会社
取引基本契約 (新規締結)
2026-07-08
新規締結
処理待ち
P12-07046495-002
V3002
大和商事株式会社
売買契約 (変更)
2026-07-16
変更
処理待ち
P12-07046495-003
V3008
沖縄物流センター
秘密保持契約 (新規締結)
2026-07-16
新規締結
承認
P12-07046495-004
V3008
沖縄物流センター
売買契約 (変更)
2026-07-18
変更
承認
P12-07046495-005
V3005
グローバルテック合同会社
取引基本契約 (新規締結)
2026-07-13
新規締結
処理待ち
P12-07046495-006
V3008
沖縄物流センター
売買契約 (変更)
2026-07-05
変更
処理待ち
P12-07046495-007
V3002
大和商事株式会社
売買契約 (変更)
2026-07-18
変更
処理待ち
P12-07046495-008
V3003
東京電子工業株式会社
取引基本契約 (新規締結)
2026-07-07
新規締結
処理待ち
P12-07046495-009
V3005
グローバルテック合同会社
業務委託契約 (解除)
2026-07-09
解除
処理待ち
P12-07046495-010
V3002
大和商事株式会社
保守委託契約 (更新)
2026-07-15
更新
処理待ち
P12-07046495-011
V3006
北陸部品工業株式会社
取引基本契